In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from pyts.multivariate.transformation import WEASELMUSE, MultivariateTransformer
from pyts.transformation import WEASEL
from pyts.image import GramianAngularField
from pyts.multivariate.image import JointRecurrencePlot
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

In [23]:
def trainScore(X,y2,y3,graphs=False,sans=False): # pass graphs and sans args as strings with file or column respectively
    X2, X3 = X,X
    
    confusion2, confusion3 = 0,0
    scores2, scores3 = [],[]

    n_iters=5
    for _ in range(n_iters):
        skf = StratifiedKFold(n_splits=5,shuffle=True) #maintains class balance in folds for 2 class case
        for train_index, test_index in skf.split(X2, y2):
            transformer1 = WEASELMUSE(word_size=5, n_bins=2, window_sizes=[12,36], chi2_threshold=5, sparse=False, strategy='uniform')
            
            X_train, X_test = X2[train_index], X2[test_index]
            y_train, y_test = y2[train_index], y2[test_index]
            #print(X2.shape,X_train.shape)

            transformer1 = transformer1.fit(X_train,y_train)
            X_train = transformer1.transform(X_train)
            clf2 = LogisticRegression(solver='newton-cg').fit(X_train,y_train)
            X_test = transformer1.transform(X_test)
            y_pred = clf2.predict(X_test)
            confusion2 += confusion_matrix(y_test, y_pred)
            scores2.append(clf2.score(X_test, y_test))
        
        for train_index, test_index in skf.split(X3, y3):
            transformer2 = WEASELMUSE(word_size=5, n_bins=2, window_sizes=[12,36], chi2_threshold=5, sparse=False, strategy='uniform') 
            X_train, X_test = X3[train_index], X3[test_index]
            y_train, y_test = y3[train_index], y3[test_index]
            transformer2 = transformer2.fit(X_train,y_train) # Fit the transformer
            X_train = transformer2.transform(X_train) # Transform the training set
            clf3 = LogisticRegression(solver='newton-cg').fit(X_train,y_train) # Fit the classifier
            X_test = transformer2.transform(X_test) # Transform the test set
            y_pred = clf3.predict(X_test) # Predict the test set labels
            confusion3 += confusion_matrix(y_test, y_pred)
            scores3.append(clf3.score(X_test, y_test)) # Append accuracy

           


        #explainer = shap.KernelExplainer(clf3.predict_proba, X_train, link="logit")
        #shap_values = explainer.shap_values(X_test, nsamples=100, l1_reg="num_features(13)")
        #shap.force_plot(explainer.expected_value[0], shap_values[0][0,:], X_test[0,:], link="logit")
    
    if sans:
        print(f"2 Class Accuracy without {sans}: {np.average(scores2):.2f}")
        print(f"3 Class Accuracy without {sans}: {np.average(scores3):.2f}")
    else:
        print(f"2 Class Accuracy: {np.average(scores2):.2f}")
        print(f"3 Class Accuracy: {np.average(scores3):.2f}")
        
    if graphs:
        if graphs=='6688':
            labels1 = ['Undistracted','6x6 and 8x8']
            labels2 = ['Undistracted','6x6','8x8']
            col = '6688'
        elif graphs=='all':
            labels1 = ['Undistracted','All Sizes']
            labels2 = ['Undistracted','4x4 and 5x5','6x6 and 8x8']
            col = 'All'
        else:
            labels1 = ['Undistracted','4x4 and 5x5']
            labels2 = ['Undistracted','4x4','5x5']
            col = '4455'
        disp2 = ConfusionMatrixDisplay(confusion_matrix=confusion2,display_labels=labels1)
        disp3 = ConfusionMatrixDisplay(confusion_matrix=confusion3,display_labels=labels2)
        disp2.plot()
        #plt.title(col)
        plt.savefig('HistPlots/2by2Confusion'+col)
        plt.show()
        disp3.plot()
        #plt.title(col)
        plt.savefig('HistPlots/3by3Confusion'+col)
        plt.show()
    return np.average(scores2), np.average(scores3)

In [3]:
def singleCol(bigDis,graphs=False):
    scores2,scores3 = [], []
    for i, col in enumerate(columns):
        score2, score3 = trainScore(X[:,i,:],y2,y3,col=col,graphs=False,bigDis=bigDis)
        scores2.append(score2)
        scores3.append(score3)
    if graphs==True:
        ax = sns.barplot(x=columns,y=scores2, order=scores2.sort())
        ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right")
        ax.set_ylabel('Accuracy')
        plt.savefig('SingleFeature/2 Class Feature Importance')
        plt.show()
        ax = sns.barplot(x=columns,y=scores3, order=scores3.sort())
        ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right")
        ax.set_ylabel('Accuracy')
        plt.savefig('SingleFeature/3 Class Feature Importance')
        plt.show()
    return scores2, scores3

In [4]:
def pairCols(bigDis):
    scores2,scores3 = {},{}
    for i, col1 in enumerate(columns):
        for j, col2 in enumerate(columns):
            if i<=j:
                continue
            cols = col1 + ' and ' + col2
            X_pair = np.concatenate((X[:,i,:].reshape(numSamples,1,-1),X[:,j,:].reshape(numSamples,1,-1)),axis=1)
            score2, score3 = trainScore(X_pair,y2,y3,col=cols,graphs=False,bigDis=bigDis)
            scores2[cols] = score2
            scores3[cols] = score3
    return scores2, scores3

In [4]:
def sansCol(X,y2,y3,scores2All,scores3All,columns):
    scores2,scores3 = {},{}
    for i, col in enumerate(columns):
        X_sans = np.delete(X,i,axis=1)
        score2, score3 = trainScore(X_sans,y2,y3,sans=col)
        scores2[col] = score2#-scores2All
        scores3[col] = score3#-scores3All
    return scores2, scores3

In [5]:
def sansPairCols(X,y2,y3,scores2All,scores3All,columns):
    scores2,scores3 = {},{}
    for i, col1 in enumerate(columns):
        for j, col2 in enumerate(columns):
            if i<=j:
                continue
            cols = col1 + ' and ' + col2
            X_sans = np.delete(X,i,axis=1)
            X_sans = np.delete(X_sans,j,axis=1)
            score2, score3 = trainScore(X_sans,y2,y3,sans=cols)
            scores2[cols] = score2#-scores2All
            scores3[cols] = score3#-scores3All
    scores2 = {k: v for k, v in sorted(scores2.items(), key=lambda item: item[1])}
    scores3 = {k: v for k, v in sorted(scores3.items(), key=lambda item: item[1])}
    return scores2, scores3

In [28]:
def modal(X,y2,y3,scores2All,scores3All,columns):
    scores2,scores3 = {},{}
    df = pd.DataFrame(X.reshape(-1,13), columns=columns)
    #df = pd.DataFrame(X.reshape(-1,12), columns=columns)
    Bio = df[['Openness','PupilL','PupilR','Objects_numeric','gsr_phasic']].values.reshape(-1,5,475)
    #Bio = df[['Openness','PupilL','PupilR','Objects_numeric']].values.reshape(-1,4,475) # uncomment for no gsr
    Kinematic = df[['Accel','Brake','Speed','Steering','Throttle']].values.reshape(-1,5,475)
    Enviro = df[['Center','Front','Back']].values.reshape(-1,3,475)
    for key, data in [['Bio',Bio], ['Kinematic',Kinematic], ['Enviro',Enviro]]:
        score2, score3 = trainScore(data,y2,y3)
        scores2[key] = score2#-scores2All
        scores3[key] = score3#-scores3All
    
    scores2 = {k: v for k, v in sorted(scores2.items(), key=lambda item: item[1])}
    scores3 = {k: v for k, v in sorted(scores3.items(), key=lambda item: item[1])}
    return scores2, scores3

In [7]:
def modalPair(X,y2,y3,scores2All,scores3All,columns):
    scores2,scores3 = {},{}
    #df = pd.DataFrame(X.reshape(-1,13), columns=columns)
    df = pd.DataFrame(X.reshape(-1,12), columns=columns)
    #Bio = df[['Openness','PupilL','PupilR','Objects_numeric','gsr_phasic']].values.reshape(-1,5,475)
    Bio = df[['Openness','PupilL','PupilR','Objects_numeric']].values.reshape(-1,4,475) # uncomment for no gsr
    Kinematic = df[['Accel','Brake','Speed','Steering','Throttle']].values.reshape(-1,5,475)
    Enviro = df[['Center','Front','Back']].values.reshape(-1,3,475)
    for key, data in [['Bio',Bio], ['Kinematic',Kinematic], ['Enviro',Enviro]]:
        score2, score3 = trainScore(data,y2,y3)
        scores2[key] = score2#-scores2All
        scores3[key] = score3#-scores3All
    
    scores2 = {k: v for k, v in sorted(scores2.items(), key=lambda item: item[1])}
    scores3 = {k: v for k, v in sorted(scores3.items(), key=lambda item: item[1])}
    return scores2, scores3

In [8]:
def loadData(type='all',new=True):
    if new:
        X4455 = pd.read_csv('newdata4455.csv')
        X6688 = pd.read_csv('newdata6688.csv')
        X = pd.concat([X4455,X6688])
        y2 = (X['State'] != 0).to_numpy().astype(int)
        y3 = []
        for i in X['State']:
            if i == 0:
                y3.append(0)
            elif i == 1 or i == 2:
                y3.append(1)
            else:
                y3.append(2)
        y3 = np.array(y3)
        State = X.pop('State')

        y2_small = []
        y3_small = []
        for num in y2:
            if len(y2_small)==0 or y2_small[-1] != num:
                y2_small.append(num)
        for num in y3:
            if len(y3_small)==0 or y3_small[-1] != num:
                y3_small.append(num)
        return X,y2_small,y3_small,State
    if type=='4455':
        X = np.load('data.npy',allow_pickle=True)
        numSamples = X.shape[0]
        y2, y3 = [0,1,1]*(numSamples//3), [0,1,2]*(numSamples//3)
        y2, y3 = np.array(y2), np.array(y3)
    elif type=='6688':
        X = np.load('data6688.npy',allow_pickle=True)
        numSamples = X.shape[0]
        y2, y3 = [0,1,1]*(numSamples//3), [0,1,2]*(numSamples//3)
        y2, y3 = np.array(y2), np.array(y3)
    elif type=='all':
        X1 = np.load('data.npy', allow_pickle=True)
        X2 = np.load('data6688.npy', allow_pickle=True)
        X = np.concatenate((X1,X2),axis=0)
        numSamples = [X1.shape[0],X2.shape[0]]
        y2 = [0,1,1]*(sum(numSamples)//3)
        y3 = [0,1,1]*(numSamples[0]//3) + [0,2,2]*(numSamples[1]//3)
        y2, y3 = np.array(y2), np.array(y3)
    return X, y2, y3

In [9]:
def meanAccuracy(resultDict,columns):
    meanAcc = {k: [] for k in columns}
    for feature in meanAcc:
        for k, v in resultDict.items():
            if feature in k:
                meanAcc[feature].append(v)
        meanAcc[feature] = np.mean(meanAcc[feature])
    meanAcc = {k: v for k, v in sorted(meanAcc.items(), key=lambda item: item[1])}
    return meanAcc

In [22]:
columns = ['Accel','Brake','Openness','PupilL','PupilR','Speed','Steering','Throttle','Center','Front','Back','Objects_numeric' ,'gsr_phasic'] # Testing without gsr_phasic
files = ['4455','6688','all']

In [ ]:
dict_scores2, dict_scores3 = {},{}
for file in files:
    if file != 'all': continue
    X, y2, y3 = loadData(file,new=False)
    X = X[:,:-1,:] # remove gsr_phasic
    scores2All, scores3All = trainScore(X,y2,y3)
    dict_scores2[file] = scores2All
    dict_scores3[file] = scores3All
    print(scores2All,scores3All)

In [ ]:
dict_scores2Sans, dict_scores3Sans = {}, {}
for file in files:
    if file != 'all': continue
    X, y2, y3 = loadData(file,new=False)
    #X = X[:,:-1,:] # remove gsr_phasic
    scores2All, scores3All = trainScore(X,y2,y3)
    scores2Sans, scores3Sans = sansCol(X,y2,y3,scores2All,scores3All,columns) # Lower number means feature is more important, number > 0 means feature is noise that is decreasing accuracy
    dict_scores2Sans[file] = meanAccuracy(scores2Sans,columns)
    dict_scores3Sans[file] = meanAccuracy(scores3Sans,columns)

In [ ]:
dict_scores2SansPair, dict_scores3SansPair = {}, {}
for file in files:
    if file != 'all': continue
    X, y2, y3 = loadData(file,new=False)
    #X = X[:,:-1,:] # remove gsr_phasic
    scores2All, scores3All = trainScore(X,y2,y3)
    scores2SansPair, scores3SansPair = sansPairCols(X,y2,y3,scores2All,scores3All,columns)
    dict_scores2SansPair[file] = meanAccuracy(scores2SansPair,columns)
    dict_scores3SansPair[file] = meanAccuracy(scores3SansPair,columns)

In [ ]:
dict_scores2Modal, dict_scores3Modal = {}, {}
for file in files:
    if file != 'all': continue
    X, y2, y3 = loadData(file,new=False)
    #X = X[:,:-1,:] # remove gsr_phasic
    scores2All, scores3All = trainScore(X,y2,y3)
    scores2Modal, scores3Modal = modal(X,y2,y3,scores2All,scores3All,columns)
    dict_scores2Modal[file] = scores2Modal # lower number means modality has lower accuracy than combined dataset
    dict_scores3Modal[file] = scores3Modal

In [ ]:
scores2, scores3 = singleCol(bigDis=file,graphs=False)
scores2, scores3 = pairCols(bigDis=file)

In [ ]:
import pickle
with open('dict_scores2Sans.pkl', 'rb') as f:
    dict_scores2Sans = pickle.load(f)

with open('dict_scores3Sans.pkl', 'rb') as f:
    dict_scores3Sans = pickle.load(f)

with open('dict_scores2SansPair.pkl', 'rb') as f:
    dict_scores2SansPair = pickle.load(f)

with open('dict_scores3SansPair.pkl', 'rb') as f:
    dict_scores3SansPair = pickle.load(f)

with open('dict_scores2Modal.pkl', 'rb') as f:
    dict_scores2Modal = pickle.load(f)

with open('dict_scores3Modal.pkl', 'rb') as f:
    dict_scores3Modal = pickle.load(f)

In [31]:
import pickle
trailing = ["nogsr.pkl","gsr.pkl"]
trailing = trailing[1]

with open('dict_scores2Sans_NEW_'+trailing, 'wb') as handle:
    pickle.dump(dict_scores2Sans, handle)
with open('dict_scores3Sans_NEW_'+trailing, 'wb') as handle:
    pickle.dump(dict_scores3Sans, handle)
with open('dict_scores2SansPair_NEW_'+trailing, 'wb') as handle:
    pickle.dump(dict_scores2SansPair, handle)
with open('dict_scores3SansPair_NEW_'+trailing, 'wb') as handle:
    pickle.dump(dict_scores3SansPair, handle)
with open('dict_scores2Modal_NEW_'+trailing, 'wb') as handle:
    pickle.dump(dict_scores2Modal, handle)
with open('dict_scores3Modal_NEW_'+trailing, 'wb') as handle:
    pickle.dump(dict_scores3Modal, handle)